####Build a machine learning model to predict whether the next day's stock closing price will be higher or lower than today's closing price. Use 5 years of historical data from Yahoo Finance for the Sensex and four selected stocks. Compare the performance of all models using evaluation metrics and business cost analysis to identify the most predictable asset for trading.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

from datetime import datetime, timedelta

In [ ]:
tickers = {
    "Sensex": "^BSESN",
    "Reliance": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFC_Bank": "HDFCBANK.NS",
    "Infosys": "INFY.NS"
}

In [ ]:
end_date = datetime.today()
start_date = end_date - timedelta(days=5*365)

stock_data = {}

for asset, ticker in tickers.items():

    print(f"Downloading {asset}...")

    df = yf.download(
        ticker,
        start=start_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        progress=False,
        auto_adjust=False
    )

    # Flatten MultiIndex columns (important for latest yfinance)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.dropna(inplace=True)

    stock_data[asset] = df

    print(f"{asset}: {len(df)} rows downloaded")

Sensex: 1231 rows downloaded
Reliance: 1237 rows downloaded
TCS: 1237 rows downloaded
HDFC_Bank: 1237 rows downloaded
Infosys: 1237 rows downloaded


In [ ]:
for asset, df in stock_data.items():
    filename = f"{asset}_5yrs.csv"
    df.to_csv(filename)
    print(f"Saved {filename}")

Saved Sensex_5yrs.csv
Saved Reliance_5yrs.csv
Saved TCS_5yrs.csv
Saved HDFC_Bank_5yrs.csv
Saved Infosys_5yrs.csv


In [ ]:
print(stock_data["Sensex"].head())

Price          Adj Close         Close          High           Low  \
Date                                                                 
2021-07-23  52975.800781  52975.800781  53114.699219  52653.769531   
2021-07-26  52852.269531  52852.269531  53103.421875  52783.628906   
2021-07-27  52578.761719  52578.761719  53024.699219  52433.179688   
2021-07-28  52443.710938  52443.710938  52673.691406  51802.730469   
2021-07-29  52653.070312  52653.070312  52777.179688  52561.390625   

Price               Open  Volume  
Date                              
2021-07-23  52967.871094   12100  
2021-07-26  52985.261719   20900  
2021-07-27  52995.718750    6300  
2021-07-28  52673.691406    8000  
2021-07-29  52693.531250    8800  


In [ ]:
for asset, df in stock_data.items():
    print(f"{asset}: {df.shape}")

Sensex: (1231, 6)
Reliance: (1237, 6)
TCS: (1237, 6)
HDFC_Bank: (1237, 6)
Infosys: (1237, 6)


In [ ]:
def prepare_data(df):

    df = df.copy()

    # 1. Daily Return
    df["Return"] = df["Close"].pct_change()

    # 2. Log Return
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    # 3. 7-Day Moving Average
    df["MA_7"] = df["Close"].rolling(window=7).mean()

    # 4. 30-Day Moving Average
    df["MA_30"] = df["Close"].rolling(window=30).mean()

    # 5. 7-Day Volatility
    df["Volatility_7"] = df["Return"].rolling(window=7).std()

    # 6. Price / MA Ratio
    df["Price_MA_Ratio"] = df["Close"] / df["MA_30"]

    # 7. High-Low Ratio
    df["High_Low_Ratio"] = df["High"] / df["Low"]

    # 8. Day of Week
    df["Day_Of_Week"] = df.index.dayofweek

    # 9. Month
    df["Month"] = df.index.month

    # 10. Target Variable
    df["Target"] = (
        df["Close"].shift(-1) > df["Close"]
    ).astype(int)

    # Remove missing rows
    df.dropna(inplace=True)

    return df

In [ ]:
prepared_data = {}

for asset, df in stock_data.items():
    prepared_data[asset] = prepare_data(df)
    print(asset, prepared_data[asset].shape)

Sensex (1202, 16)
Reliance (1208, 16)
TCS (1208, 16)
HDFC_Bank (1208, 16)
Infosys (1208, 16)


In [ ]:
prepared_data["Sensex"].head()

Price,Adj Close,Close,High,Low,Open,Volume,Return,Log_Return,MA_7,MA_30,Volatility_7,Price_MA_Ratio,High_Low_Ratio,Day_Of_Week,Month,Target
Date,,,,,,,,,,,,,,,,
2021-09-03,58129.949219,58129.949219,58194.789062,57764.070312,57983.449219,6100,0.004795,0.004784,57119.524554,54979.959635,0.006276,1.057293,1.007457,4,9,1
2021-09-06,58296.910156,58296.910156,58515.851562,58200.289062,58411.621094,5600,0.002872,0.002868,57454.925781,55157.329948,0.005956,1.056920,1.005422,0,9,0
2021-09-07,58279.480469,58279.480469,58553.070312,58005.070312,58418.691406,7400,-0.000299,-0.000299,57762.748884,55338.236979,0.006350,1.053150,1.009447,1,9,0
2021-09-08,58250.261719,58250.261719,58372.941406,57924.480469,58350.558594,5300,-0.000501,-0.000501,57957.106027,55527.286979,0.005491,1.049038,1.007742,2,9,1
2021-09-09,58305.070312,58305.070312,58334.589844,58084.988281,58172.980469,5300,0.000941,0.000940,58064.631696,55722.665625,0.004132,1.046344,1.004297,3,9,0


In [ ]:
for asset, df in prepared_data.items():
    print(f"\n{asset}")
    print(df.isnull().sum())


Sensex
Price
Adj Close         0
Close             0
High              0
Low               0
Open              0
Volume            0
Return            0
Log_Return        0
MA_7              0
MA_30             0
Volatility_7      0
Price_MA_Ratio    0
High_Low_Ratio    0
Day_Of_Week       0
Month             0
Target            0
dtype: int64

Reliance
Price
Adj Close         0
Close             0
High              0
Low               0
Open              0
Volume            0
Return            0
Log_Return        0
MA_7              0
MA_30             0
Volatility_7      0
Price_MA_Ratio    0
High_Low_Ratio    0
Day_Of_Week       0
Month             0
Target            0
dtype: int64

TCS
Price
Adj Close         0
Close             0
High              0
Low               0
Open              0
Volume            0
Return            0
Log_Return        0
MA_7              0
MA_30             0
Volatility_7      0
Price_MA_Ratio    0
High_Low_Ratio    0
Day_Of_Week       0
Month        

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

In [ ]:
feature_columns = [
    "Return",
    "Log_Return",
    "MA_7",
    "MA_30",
    "Volatility_7",
    "Price_MA_Ratio",
    "High_Low_Ratio",
    "Day_Of_Week",
    "Month"
]

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [ ]:
results = {}
predictions = {}
trained_models = {}
test_sets = {}

In [ ]:
for asset, df in prepared_data.items():

    print(f"\nTraining model for {asset}")

    X = df[feature_columns]

    y = df["Target"]

    split = int(len(df) * 0.80)

    X_train = X.iloc[:split]

    X_test = X.iloc[split:]

    y_train = y.iloc[:split]

    y_test = y.iloc[split:]

    # Time Series Cross Validation
    tscv = TimeSeriesSplit(n_splits=5)

    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring="accuracy"
    )
    print("Cross Validation Accuracy:", cv_scores.mean())

    # Train Final Model
    pipeline.fit(X_train, y_train)

    # Predictions
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Store everything
    trained_models[asset] = pipeline
    predictions[asset] = (y_test, y_pred, y_prob)
    test_sets[asset] = X_test

    # Metrics
    results[asset] = {
        "CV Accuracy": cv_scores.mean(),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob)
    }


Training model for Sensex
Cross Validation Accuracy: 0.49124999999999996

Training model for Reliance
Cross Validation Accuracy: 0.501863354037267

Training model for TCS
Cross Validation Accuracy: 0.4894409937888199

Training model for HDFC_Bank
Cross Validation Accuracy: 0.5142857142857143

Training model for Infosys
Cross Validation Accuracy: 0.49440993788819876


In [ ]:
results_df = pd.DataFrame(results).T
results_df

,CV Accuracy,Accuracy,Precision,Recall,F1 Score,ROC AUC
Sensex,0.491250,0.539419,0.569767,0.398374,0.468900,0.559081
Reliance,0.501863,0.491736,0.475410,0.495726,0.485356,0.482735
TCS,0.489441,0.458678,0.446078,0.834862,0.581470,0.495723
HDFC_Bank,0.514286,0.487603,0.411290,0.500000,0.451327,0.490721
Infosys,0.494410,0.483471,0.435897,0.647619,0.521073,0.448210


In [ ]:
results_df.to_csv("Model_Performance.csv")
print("Performance table saved successfully.")

Performance table saved successfully.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
for asset in predictions.keys():

    y_test, y_pred, y_prob = predictions[asset]
    cm = confusion_matrix(y_test, y_pred)

    fig = px.imshow(
        cm,
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predicted", y="Actual", color="Count"),
        x=["Down (0)", "Up (1)"],
        y=["Down (0)", "Up (1)"],
        title=f"Confusion Matrix - {asset}"
    )

    fig.show()

In [ ]:
for asset in predictions.keys():

    y_test, y_pred, y_prob = predictions[asset]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    auc = roc_auc_score(y_test, y_prob)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"AUC = {auc:.3f}"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0,1],
            y=[0,1],
            mode="lines",
            name="Random Guess",
            line=dict(dash="dash")
        )
    )

    fig.update_layout(
        title=f"ROC Curve - {asset}",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate"
    )

    fig.show()

In [ ]:
for asset in predictions.keys():

    y_test, y_pred, y_prob = predictions[asset]

    precision, recall, _ = precision_recall_curve(
        y_test,
        y_prob
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=asset
        )
    )

    fig.update_layout(
        title=f"Precision-Recall Curve - {asset}",
        xaxis_title="Recall",
        yaxis_title="Precision"
    )

    fig.show()

In [ ]:
for asset in trained_models.keys():

    model = trained_models[asset].named_steps["model"]
    importance = model.feature_importances_
    importance_df = pd.DataFrame({
        "Feature": feature_columns,
        "Importance": importance
    })

    importance_df = importance_df.sort_values(
        by="Importance",
        ascending=False
    )

    fig = px.bar(
        importance_df,
        x="Feature",
        y="Importance",
        title=f"Feature Importance - {asset}",
        text_auto=".3f"
    )

    fig.show()

In [ ]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"
]

comparison_df = results_df.reset_index()
comparison_df.rename(
    columns={"index":"Asset"},
    inplace=True
)

In [ ]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"
]

comparison_df = results_df.reset_index()

comparison_df.rename(
    columns={"index":"Asset"},
    inplace=True
)

In [ ]:
comparison_long = comparison_df.melt(
    id_vars="Asset",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    comparison_long,
    x="Metric",
    y="Score",
    color="Asset",
    barmode="group",
    text_auto=".3f",
    title="Performance Comparison of All Assets"
)
fig.show()

In [ ]:
import numpy as np

In [ ]:
def calculate_cost(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = (fp * 100) + (fn * 50)
    return total_cost

In [ ]:
threshold_results = []

optimal_thresholds = {}

for asset in predictions.keys():
    y_test, y_pred, y_prob = predictions[asset]
    thresholds = np.arange(0.1, 0.91, 0.05)
    costs = []
    best_threshold = None
    minimum_cost = float("inf")

    for threshold in thresholds:

        pred = (y_prob >= threshold).astype(int)
        cost = calculate_cost(y_test, pred)
        costs.append(cost)
        if cost < minimum_cost:
            minimum_cost = cost
            best_threshold = threshold

    optimal_thresholds[asset] = {
        "Threshold": best_threshold,
        "Minimum Cost": minimum_cost
    }

    threshold_results.append({
        "Asset": asset,
        "Thresholds": thresholds,
        "Costs": costs
    })

In [ ]:
for result in threshold_results:

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=result["Thresholds"],
            y=result["Costs"],
            mode="lines+markers",
            name=result["Asset"]
        )
    )

    fig.update_layout(
        title=f"Business Cost vs Threshold - {result['Asset']}",
        xaxis_title="Threshold",
        yaxis_title="Business Cost ($)"
    )

    fig.show()

In [ ]:
cost_df = pd.DataFrame(optimal_thresholds).T
cost_df

,Threshold,Minimum Cost
Sensex,0.85,6150.0
Reliance,0.80,5850.0
TCS,0.80,5450.0
HDFC_Bank,0.75,5100.0
Infosys,0.85,5250.0


In [ ]:
cost_df.to_csv("Business_Cost_Analysis.csv")
print("Business Cost Analysis saved.")

Business Cost Analysis saved.


In [ ]:
best_asset = cost_df["Minimum Cost"].idxmin()
print("Best Asset Based on Business Cost:", best_asset)
print(cost_df.loc[best_asset])

Best Asset Based on Business Cost: HDFC_Bank
Threshold          0.75
Minimum Cost    5100.00
Name: HDFC_Bank, dtype: float64


In [ ]:
#merging performance and cost tables
final_results = results_df.copy()
final_results["Optimal Threshold"] = cost_df["Threshold"]
final_results["Minimum Business Cost"] = cost_df["Minimum Cost"]
final_results = final_results.round(3)
final_results

,CV Accuracy,Accuracy,Precision,Recall,F1 Score,ROC AUC,Optimal Threshold,Minimum Business Cost
Sensex,0.491,0.539,0.570,0.398,0.469,0.559,0.85,6150.0
Reliance,0.502,0.492,0.475,0.496,0.485,0.483,0.80,5850.0
TCS,0.489,0.459,0.446,0.835,0.581,0.496,0.80,5450.0
HDFC_Bank,0.514,0.488,0.411,0.500,0.451,0.491,0.75,5100.0
Infosys,0.494,0.483,0.436,0.648,0.521,0.448,0.85,5250.0


In [ ]:
final_results.to_csv("Final_Results.csv")
print("Final Results saved successfully.")

Final Results saved successfully.


In [ ]:
best_accuracy = final_results["Accuracy"].idxmax()
print("Best Asset by Accuracy")
print(best_accuracy)
print(final_results.loc[best_accuracy])

Best Asset by Accuracy
Sensex
CV Accuracy                 0.491
Accuracy                    0.539
Precision                   0.570
Recall                      0.398
F1 Score                    0.469
ROC AUC                     0.559
Optimal Threshold           0.850
Minimum Business Cost    6150.000
Name: Sensex, dtype: float64


In [ ]:
best_auc = final_results["ROC AUC"].idxmax()
print("Best Asset by ROC-AUC")
print(best_auc)
print(final_results.loc[best_auc])

Best Asset by ROC-AUC
Sensex
CV Accuracy                 0.491
Accuracy                    0.539
Precision                   0.570
Recall                      0.398
F1 Score                    0.469
ROC AUC                     0.559
Optimal Threshold           0.850
Minimum Business Cost    6150.000
Name: Sensex, dtype: float64


In [ ]:
best_cost = final_results["Minimum Business Cost"].idxmin()
print("Best Asset by Business Cost")
print(best_cost)
print(final_results.loc[best_cost])

Best Asset by Business Cost
HDFC_Bank
CV Accuracy                 0.514
Accuracy                    0.488
Precision                   0.411
Recall                      0.500
F1 Score                    0.451
ROC AUC                     0.491
Optimal Threshold           0.750
Minimum Business Cost    5100.000
Name: HDFC_Bank, dtype: float64


In [ ]:
feature_importance = pd.DataFrame(index=feature_columns)

for asset in trained_models.keys():
    model = trained_models[asset].named_steps["model"]
    feature_importance[asset] = model.feature_importances_
feature_importance["Average"] = feature_importance.mean(axis=1)

feature_importance = feature_importance.sort_values(
    by="Average",
    ascending=False
)
feature_importance

,Sensex,Reliance,TCS,HDFC_Bank,Infosys,Average
Price_MA_Ratio,0.133898,0.133898,0.133898,0.133898,0.133898,0.133898
MA_7,0.132906,0.132906,0.132906,0.132906,0.132906,0.132906
High_Low_Ratio,0.131193,0.131193,0.131193,0.131193,0.131193,0.131193
Volatility_7,0.129246,0.129246,0.129246,0.129246,0.129246,0.129246
MA_30,0.125417,0.125417,0.125417,0.125417,0.125417,0.125417
Log_Return,0.117884,0.117884,0.117884,0.117884,0.117884,0.117884
Return,0.115787,0.115787,0.115787,0.115787,0.115787,0.115787
Month,0.062943,0.062943,0.062943,0.062943,0.062943,0.062943
Day_Of_Week,0.050725,0.050725,0.050725,0.050725,0.050725,0.050725


In [ ]:
fig = px.bar(
    feature_importance.reset_index(),
    x="index",
    y="Average",
    text_auto=".3f",
    title="Average Feature Importance Across All Assets"
)

fig.update_layout(
    xaxis_title="Feature",
    yaxis_title="Average Importance"
)

fig.show()

In [ ]:
display(final_results)

,CV Accuracy,Accuracy,Precision,Recall,F1 Score,ROC AUC,Optimal Threshold,Minimum Business Cost
Sensex,0.491,0.539,0.570,0.398,0.469,0.559,0.85,6150.0
Reliance,0.502,0.492,0.475,0.496,0.485,0.483,0.80,5850.0
TCS,0.489,0.459,0.446,0.835,0.581,0.496,0.80,5450.0
HDFC_Bank,0.514,0.488,0.411,0.500,0.451,0.491,0.75,5100.0
Infosys,0.494,0.483,0.436,0.648,0.521,0.448,0.85,5250.0


In [ ]:
print("=" * 60)
print("FINAL RECOMMENDATION")
print("=" * 60)
print(f"Best Accuracy           : {best_accuracy}")
print(f"Best ROC-AUC            : {best_auc}")
print(f"Lowest Business Cost    : {best_cost}")
print("\nRecommended Asset for Trading:")
print(best_cost)
print("=" * 60)

FINAL RECOMMENDATION
Best Accuracy           : Sensex
Best ROC-AUC            : Sensex
Lowest Business Cost    : HDFC_Bank

Recommended Asset for Trading:
HDFC_Bank
